# 02 — Baseline Modeling

**Today's ticket: `HC-M1-05` — validation strategy.** (`HC-M1-06` dummy baseline, `HC-M1-07` logistic regression baseline, `HC-M1-08` experiment comparison will build on this split in later sections of this same notebook.)

Scope for the split, per [`docs/problem_definition.md`](../docs/problem_definition.md) and `notebooks/01_data_understanding.ipynb`:

- Features for the baseline come from `application_train` only — the bureau/previous-credit tables are feature-engineering scope for a later milestone, not this one.
- `TARGET` is ~11.4:1 imbalanced (confirmed in notebook 01), so the split **must be stratified** — an unstratified split risks a validation fold with a meaningfully different positive rate, which would make validation ROC-AUC an unreliable estimate.
- The split happens once, here, and is saved to disk. Every later notebook/script reads the same saved split rather than re-splitting, so "did you reproduce my result" has a concrete, checkable answer (`HC-M1-09`).

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2

CACHE_DB = Path("../reports/data_profile/.landing_cache.duckdb")
SPLIT_PATH = Path("../data/interim/train_valid_split.csv")

con = duckdb.connect(str(CACHE_DB), read_only=True)
application_train = con.sql("SELECT * FROM application_train").df()
con.close()

n_rows, n_cols = application_train.shape
print(f"application_train: {n_rows:,} rows, {n_cols} columns")

application_train: 307,511 rows, 122 columns


## Define X, y

`SK_ID_CURR` is an identifier, not a feature — it's kept alongside the split (not fed to any model) so the split is auditable at the row level and so later notebooks/scripts can rejoin predictions back to applicants. `TARGET` is the label. Everything else in `application_train` is a candidate feature for the baseline.

In [2]:
ids = application_train["SK_ID_CURR"]
y = application_train["TARGET"]
X = application_train.drop(columns=["SK_ID_CURR", "TARGET"])

print(f"X: {X.shape}, y: {y.shape}")
print(f"Positive rate in full application_train: {y.mean():.4f}")

X: (307511, 120), y: (307511,)
Positive rate in full application_train: 0.0807


## Stratified train/validation split

Parameter choices, stated explicitly (this is exactly the kind of thing worth being able to justify in an interview, not just cite):

- **`test_size=0.2`** — a conventional 80/20 split; with 307k rows, 20% (~61.5k) is more than enough to estimate ROC-AUC precisely, so there's no need to trade away training data for a larger validation set.
- **`stratify=y`** — forces the same ~8.07% positive rate in both the training and validation folds. Without this, a random split could by chance produce a validation fold with a noticeably different imbalance ratio, and any ROC-AUC measured on it would be a noisier, less trustworthy estimate of real generalization.
- **`random_state=42`** — fixed and recorded here explicitly, not left to a default. This is what makes "reproduce my baseline result" a well-posed request instead of a matter of luck (`HC-M1-09`).
- **The validation set (`X_valid`, `y_valid`) is not touched again until scoring.** No fitting, no imputation, no encoding, no hyperparameter selection ever sees it before the final `.predict_proba()` call in each experiment below.

In [3]:
X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
    X,
    y,
    ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}")
print(f"Train positive rate:      {y_train.mean():.4f}")
print(f"Validation positive rate: {y_valid.mean():.4f}")

X_train: (246008, 120), X_valid: (61503, 120)
Train positive rate:      0.0807
Validation positive rate: 0.0807


## Persist the split

Saved as `SK_ID_CURR -> split` rather than re-deriving it from `random_state` alone: it's directly inspectable (open the CSV, see exactly which applicants are in which fold), it's what a teammate or a future notebook actually joins against to reproduce results, and it decouples "did we get the same split" from "does everyone have scikit-learn installed with identical version behavior."

In [4]:
split_assignment = pd.concat(
    [
        pd.DataFrame({"SK_ID_CURR": ids_train, "split": "train"}),
        pd.DataFrame({"SK_ID_CURR": ids_valid, "split": "valid"}),
    ],
    ignore_index=True,
).sort_values("SK_ID_CURR")

SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
split_assignment.to_csv(SPLIT_PATH, index=False)

print(f"Wrote {SPLIT_PATH} ({len(split_assignment):,} rows)")
split_assignment["split"].value_counts()

Wrote ../data/interim/train_valid_split.csv (307,511 rows)


split
train    246008
valid     61503
Name: count, dtype: int64

## Summary — HC-M1-05 acceptance criteria

- [x] Reproducible split (`random_state=42`, saved to `data/interim/train_valid_split.csv`)
- [x] Stratification used (`stratify=y`)
- [x] `random_state` documented (42, stated inline with rationale above)
- [x] Validation set untouched during training (no fitting/imputation/encoding has happened yet — that starts in the next section, `HC-M1-07`, strictly on `X_train`)
- [ ] ROC-AUC calculated on validation predictions — blocked on having a model; comes next with the dummy baseline (`HC-M1-06`)

Next: `HC-M1-06` — dummy baseline, to establish the ROC-AUC floor (~0.5) this split will be scored against.